# DiCo-NLI - ModernBERT Baseline Testing

**Repository:** https://github.com/AmnO-O/DICO-NLI

Fine-tunes **`answerdotai/ModernBERT-base`** and **`answerdotai/ModernBERT-large`** on the
official DiCo-NLI **train** split and evaluates on the official **dev** split with the
**official scorer** (`evaluation_functions`), across:

- seeds: `2021, 2022, 2023, 2024, 2025`
- tracks: `1 (EN), 2 (ES), 3 (EU), 4 (mixed)`

That is `2 models x 4 tracks x 5 seeds = 40` runs.

## How it works

Each run is launched as a subprocess of `starter_kit/main.py` (the documented pipeline), so
the notebook reproduces the starter-kit protocol exactly. After each run the notebook:

1. picks the **last epoch** dev prediction file (`predictions_dev_<timestamp>.csv`),
2. scores it against `<DATA_ROOT>/dev/dico_nli_dev_track<t>_reference.csv` with the official
   scorer, and
3. appends `weighted_f1`, `soft_cons`, `hard_cons` (+ diagnostics) to a running CSV.

Runs are **resumable**: re-running the notebook skips experiments that already have
predictions on disk. Set `OVERWRITE = True` to force a clean re-run.

## Data location (auto-detected)

The setup cell resolves two roots automatically:

- **`REPO_ROOT`** - the code checkout (`starter_kit/`, `evaluation_functions/`).
- **`DATA_ROOT`** - the data, which must contain `train/` and `dev/`.

| Environment | `DATA_ROOT` |
|-------------|-------------|
| Kaggle | `/kaggle/input/datasets/ieltsmater/dico-nli/DiCO-NLI` |
| Local checkout | `<repo>/final_data` |

## Running on Kaggle

1. Add the `ieltsmater/dico-nli` dataset to the notebook.
2. Enable **Internet** (Settings -> Internet on) so the setup cell can `git clone` the repo
   into `/kaggle/working/DiCO-NLI` if it is not already present.
3. Run all cells; outputs land in `/kaggle/working/results/baselines_modernbert`.

## Running locally

```bash
pip install -r starter_kit/requirements.txt
```

GPU strongly recommended (ModernBERT-large is heavy). If you only want a quick smoke of the
notebook, reduce `SEEDS`, `TRACKS`, or `MODELS` in the config cell below.

## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
import shutil
import subprocess
import sys
import time
import traceback
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:  # plain-python fallback (nbconvert / script)
    def display(obj):
        print(obj)


GITHUB_REPO = "https://github.com/AmnO-O/DICO-NLI"
GITHUB_BRANCH = "main"

# Kaggle paths (only active when running on Kaggle's Linux environment).
KAGGLE_WORKING = Path("/kaggle/working")
KAGGLE_DATA_ROOT = Path("/kaggle/input/datasets/ieltsmater/dico-nli/DiCO-NLI")
IS_KAGGLE = sys.platform.startswith("linux") and KAGGLE_WORKING.is_dir()

# Set to a path to force a specific data directory, or leave None to auto-detect.
DATA_ROOT_OVERRIDE = None


def is_repo_root(path: Path) -> bool:
    return (path / "starter_kit" / "main.py").exists() and (path / "evaluation_functions").is_dir()


def find_repo_root(start: Path):
    for candidate in [start, *start.parents]:
        if is_repo_root(candidate):
            return candidate
    return None


def ensure_repo() -> Path:
    # 1) a checkout is already available (local machine, or bundled in the Kaggle dataset)
    candidates = [Path.cwd().resolve()]
    if IS_KAGGLE:
        candidates += [KAGGLE_DATA_ROOT, KAGGLE_WORKING / "DiCO-NLI"]
    for candidate in candidates:
        if candidate.exists():
            root = find_repo_root(candidate)
            if root is not None:
                return root
    # 2) otherwise clone it (Kaggle: enable Internet in the notebook settings)
    clone_dir = (KAGGLE_WORKING if IS_KAGGLE else Path.cwd()) / "DiCO-NLI"
    if not is_repo_root(clone_dir):
        print(f"Cloning {GITHUB_REPO} -> {clone_dir}")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO, str(clone_dir)],
            check=True,
        )
    return clone_dir


def resolve_data_root() -> Path:
    if DATA_ROOT_OVERRIDE is not None:
        return Path(DATA_ROOT_OVERRIDE)
    if IS_KAGGLE and (KAGGLE_DATA_ROOT / "train").is_dir() and (KAGGLE_DATA_ROOT / "dev").is_dir():
        return KAGGLE_DATA_ROOT
    return REPO_ROOT / "final_data"


REPO_ROOT = ensure_repo()
STARTER_KIT = REPO_ROOT / "starter_kit"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evaluation_functions import score_files  # noqa: E402

DATA_ROOT = resolve_data_root()

print("Python  :", sys.executable)
print("Kaggle  :", IS_KAGGLE)
print("Repo    :", REPO_ROOT)
print("GitHub  :", GITHUB_REPO)
print("Data    :", DATA_ROOT)
print("Train   :", (DATA_ROOT / "train").exists())
print("Dev     :", (DATA_ROOT / "dev").exists())

def run_logged_command(cmd, cwd, log_path, echo=("Epoch", "average loss", "Training history")):
    # Run a command, tee its stdout/stderr into log_path, and echo the lines that
    # matter (per-epoch loss) live so you can watch training progress.
    with Path(log_path).open("w", encoding="utf-8") as log:
        log.write(" ".join(cmd) + "\n\n")
        log.flush()
        proc = subprocess.Popen(
            cmd,
            cwd=str(cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            log.write(line)
            log.flush()
            if any(token in line for token in echo):
                print("    " + line.rstrip())
        returncode = proc.wait()
    return returncode


In [ ]:
# ----------------------------- experiment grid -----------------------------
MODELS = {
    "ModernBERT-base": "answerdotai/ModernBERT-base",
    "ModernBERT-large": "answerdotai/ModernBERT-large",
}
SEEDS = [2021, 2022, 2023, 2024, 2025]
TRACKS = [1, 2, 3, 4]

# ----------------------------- hyperparameters -----------------------------
# These must satisfy starter_kit/main.py validation rules.
MAX_LENGTH = 256
NUM_EPOCHS = 3
LR = 5e-5          # allowed: 1e-6, 5e-6, 8e-6, 9e-6, 1e-5, 1.5e-5, 2e-5, 2.5e-5, 3e-5, 4e-5, 5e-5, 8e-5, 1e-4
BATCH_SIZE = 32    # multiple of 8, 8 <= batch_size <= 64
WD = 1e-5          # allowed: 1e-6, 5e-6, 8e-6, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2, 1e-1
WARMUP_PCT = 0.01  # 0 .. 0.1, step 0.01
DROPOUT = 0.1      # 0 .. 0.5

# ----------------------------- output locations ----------------------------
# On Kaggle write under /kaggle/working so results are kept as notebook output.
OUTPUT_ROOT = KAGGLE_WORKING if IS_KAGGLE else REPO_ROOT
RESULTS_DIR = OUTPUT_ROOT / "results" / "baselines_modernbert"
LOG_DIR = RESULTS_DIR / "logs"
SUMMARY_CSV = RESULTS_DIR / "baseline_scores.csv"
OVERWRITE = False  # True -> delete and re-run completed experiments

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

TOTAL = len(MODELS) * len(TRACKS) * len(SEEDS)
print(f"Configured {TOTAL} experiments -> {RESULTS_DIR}")

## 2. Helper functions

In [ ]:
def track_paths(track: int):
    train = DATA_ROOT / "train" / f"dico_nli_train_track{track}_participant_labeled.csv"
    dev = DATA_ROOT / "dev" / f"dico_nli_dev_track{track}_participant_labeled.csv"
    ref = DATA_ROOT / "dev" / f"dico_nli_dev_track{track}_reference.csv"
    for path in (train, dev, ref):
        if not path.exists():
            raise FileNotFoundError(path)
    return train, dev, ref


def run_key(model_key: str, track: int, seed: int) -> str:
    slug = model_key.lower().replace(".", "")
    return f"{slug}_track{track}_seed{seed}"


def build_command(model_id, train, dev, ref, seed, results_dir):
    return [
        sys.executable, "-u", "main.py",
        "--model_path", model_id,
        "--tokenizer_path", model_id,
        "--do_train",
        "--train_path", str(train),
        "--do_dev",
        "--dev_path", str(dev),
        "--no-do_test",
        "--seed", str(seed),
        "--dropout", str(DROPOUT),
        "--warmup_pcrt", str(WARMUP_PCT),
        "--lr", str(LR),
        "--batch_size", str(BATCH_SIZE),
        "--wd", str(WD),
        "--num_epochs", str(NUM_EPOCHS),
        "--no-is_optuna_trial",
        "--max_length", str(MAX_LENGTH),
        "--results_dir", str(results_dir),
        "--reference_path", str(ref),
    ]


def latest_dev_predictions(run_dir: Path) -> Path:
    preds = sorted(run_dir.glob("predictions_dev_*.csv"))
    if not preds:
        raise FileNotFoundError(f"No dev predictions found in {run_dir}")
    return preds[-1]


def score_predictions(key, model_key, model_id, track, seed, pred_path, ref_path, run_dir, elapsed=None):
    report = score_files(ref_path, pred_path, output_dir=run_dir / "official_scores")
    return {
        "run": key,
        "model": model_key,
        "model_id": model_id,
        "track": track,
        "seed": seed,
        "weighted_f1": report.weighted_f1,
        "soft_cons": report.soft_cons,
        "hard_cons": report.hard_cons,
        "macro_f1": report.classification.macro_f1,
        "accuracy": report.classification.accuracy,
        "instances": report.classification.total_instances,
        "predictions": str(pred_path),
        "minutes": None if elapsed is None else round(elapsed / 60, 2),
    }


def run_experiment(model_key, model_id, track, seed, *, overwrite=OVERWRITE):
    train, dev, ref = track_paths(track)
    key = run_key(model_key, track, seed)
    run_dir = RESULTS_DIR / key
    log_path = LOG_DIR / f"{key}.log"

    if run_dir.exists() and not overwrite:
        preds = sorted(run_dir.glob("predictions_dev_*.csv"))
        if preds:
            print(f"[skip] {key} -> scoring existing predictions")
            return score_predictions(key, model_key, model_id, track, seed, preds[-1], ref, run_dir)

    if overwrite and run_dir.exists():
        shutil.rmtree(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    cmd = build_command(model_id, train, dev, ref, seed, run_dir)
    print(f"[run ] {key}")
    started = time.time()
    returncode = run_logged_command(cmd, STARTER_KIT, log_path)
    elapsed = time.time() - started

    if returncode != 0:
        print(f"[fail] {key}: exit {returncode}; see {log_path}")
        return None

    pred_path = latest_dev_predictions(run_dir)
    print(f"[ok  ] {key} in {elapsed / 60:.1f} min -> {pred_path.name}")
    return score_predictions(key, model_key, model_id, track, seed, pred_path, ref, run_dir, elapsed)

## 3. Run the experiment grid

In [ ]:
if SUMMARY_CSV.exists() and not OVERWRITE:
    results = pd.read_csv(SUMMARY_CSV).to_dict("records")
    done = {(r["model"], int(r["track"]), int(r["seed"])) for r in results}
else:
    results, done = [], set()

print(f"{len(done)}/{TOTAL} experiments already scored\n")

for model_key, model_id in MODELS.items():
    for track in TRACKS:
        for seed in SEEDS:
            if (model_key, track, seed) in done:
                continue
            try:
                row = run_experiment(model_key, model_id, track, seed)
            except Exception:
                traceback.print_exc()
                row = None
            if row is not None:
                results.append(row)
                done.add((model_key, track, seed))
                pd.DataFrame(results).to_csv(SUMMARY_CSV, index=False)
                print(
                    f"     -> weighted_f1={row['weighted_f1']:.4f}  "
                    f"soft_cons={row['soft_cons']:.4f}  hard_cons={row['hard_cons']:.4f}"
                )

df = pd.DataFrame(results)
print(f"\nFinished: {len(df)}/{TOTAL} experiments scored.")

## 4. Aggregated results

In [ ]:
df = pd.read_csv(SUMMARY_CSV)
df["track"] = df["track"].astype(int)
df["seed"] = df["seed"].astype(int)

summary = (
    df.groupby(["model", "track"], as_index=False)
    .agg(
        n=("seed", "count"),
        weighted_f1_mean=("weighted_f1", "mean"),
        weighted_f1_std=("weighted_f1", "std"),
        soft_cons_mean=("soft_cons", "mean"),
        soft_cons_std=("soft_cons", "std"),
        hard_cons_mean=("hard_cons", "mean"),
        hard_cons_std=("hard_cons", "std"),
    )
)
summary.to_csv(RESULTS_DIR / "baseline_summary_by_model_track.csv", index=False)
print("Per model x track (mean +/- std over seeds):")
display(summary.round(4))

macro = (
    summary.groupby("model", as_index=False)
    .agg(
        weighted_f1=("weighted_f1_mean", "mean"),
        soft_cons=("soft_cons_mean", "mean"),
        hard_cons=("hard_cons_mean", "mean"),
    )
)
macro.to_csv(RESULTS_DIR / "baseline_summary_macro.csv", index=False)
print("\nMacro-average across tracks:")
display(macro.round(4))

In [ ]:
print("weighted_f1 by seed:")
display(df.pivot_table(index=["model", "track"], columns="seed", values="weighted_f1").round(4))

print("\nhard_cons by seed:")
display(df.pivot_table(index=["model", "track"], columns="seed", values="hard_cons").round(4))

## 5. Training curves (per-epoch loss)

In [ ]:
# Every training run writes a per-epoch history to <run_dir>/training_history.json
# (and a matching .csv). Because this notebook trains with --do_dev, the history also
# contains the dev metrics per epoch - so you can watch both loss and dev F1.
def load_history(run_dir):
    path = Path(run_dir) / "training_history.json"
    if not path.exists():
        return None
    return pd.DataFrame(json.loads(path.read_text(encoding="utf-8")))


def find_histories():
    return sorted(RESULTS_DIR.glob("*/training_history.json"))


available = find_histories()
print(f"{len(available)} run(s) with a recorded training history:")
for path in available:
    print(" -", path.parent.name)

In [ ]:
# Inspect a single run's learning curve (defaults to the first available run).
RUN_TO_PLOT = available[0].parent.name if available else None
print("Selected:", RUN_TO_PLOT)

hist = load_history(RESULTS_DIR / RUN_TO_PLOT) if RUN_TO_PLOT else None
if hist is None:
    print("No training history yet - run the training grid first.")
else:
    display(hist)
    try:
        import matplotlib.pyplot as plt

        fig, ax1 = plt.subplots(figsize=(7, 4))
        ax1.plot(hist["epoch"], hist["train_loss"], marker="o", color="tab:red")
        ax1.set_xlabel("epoch")
        ax1.set_ylabel("train loss", color="tab:red")
        ax1.tick_params(axis="y", labelcolor="tab:red")
        if "dev_f1_weighted" in hist.columns:
            ax2 = ax1.twinx()
            ax2.plot(hist["epoch"], hist["dev_f1_weighted"], marker="s", color="tab:blue")
            ax2.set_ylabel("dev weighted F1", color="tab:blue")
            ax2.tick_params(axis="y", labelcolor="tab:blue")
        ax1.set_title(f"{RUN_TO_PLOT}: loss curve")
        fig.tight_layout()
        plt.show()
    except ImportError:
        print("(matplotlib not installed - the table above has the per-epoch values)")

In [ ]:
# Overlay every run's train loss, and show the raw log tail of a chosen run.
if available:
    try:
        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(figsize=(9, 5))
        for path in available:
            h = load_history(path.parent)
            ax.plot(h["epoch"], h["train_loss"], marker="o", markersize=3, label=path.parent.name)
        ax.set_xlabel("epoch")
        ax.set_ylabel("train loss")
        ax.set_title("Train loss per run")
        ax.legend(fontsize=7)
        plt.show()
    except ImportError:
        print("(matplotlib not installed)")

RUN_TO_INSPECT = available[0].parent.name if available else None
log_path = LOG_DIR / f"{RUN_TO_INSPECT}_train.log" if RUN_TO_INSPECT else None
if log_path is not None and log_path.exists():
    lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    print(f"\n{log_path.name} ({len(lines)} lines) - last 40:")
    print("\n".join(lines[-40:]))
else:
    print("No training log yet:", log_path)

## 6. Inspect the best run in detail

In [ ]:
best = df.sort_values(["hard_cons", "weighted_f1"], ascending=False).iloc[0]
run_dir = Path(best["predictions"]).parent
scores = json.loads((run_dir / "official_scores" / "scores.json").read_text(encoding="utf-8"))

print("Run       :", run_dir.name)
print("Weighted F1:", round(scores["weighted_f1"], 4))
print("SoftCons   :", round(scores["soft_cons"], 4))
print("HardCons   :", round(scores["hard_cons"], 4))

print("\nPer-label metrics:")
display(pd.DataFrame(scores["classification"]["per_label"]).T)

print("\nConfusion matrix (rows = gold, cols = predicted):")
display(pd.DataFrame(scores["classification"]["confusion_matrix"]))

print("\nSample pair errors (first 10):")
display(pd.DataFrame(scores["consistency"]["pair_errors"]).head(10))

## 7. (Optional) Tunable single run

Use this cell to try one configuration without going through the whole grid.

In [ ]:
# Example: single ModernBERT-large run on track 4, seed 2021
MODEL_KEY = "ModernBERT-large"
TRACK = 4
SEED = 2021

row = run_experiment(MODEL_KEY, MODELS[MODEL_KEY], TRACK, SEED, overwrite=True)
pd.Series(row).to_frame().T